In [1]:
import pandas as pd

# reading dataset

read the amazon review dataset
selected specific columns like 'reviewText','overall'
coverted overall from numeric like 1,2,3,4,5 to positive and negative
i am gonna use this two columns rest of my project for sentiment analysis

In [ ]:
df = pd.read_csv('amazon_reviews.csv')

df = df[['reviewText','overall']]

def rating_to_sentiment(rating):
        if(rating>3):
                return 'positive'
        elif(rating<=3):
                return 'negative'

df['overall'] = df['overall'].apply(rating_to_sentiment)

0    positive
1    positive
2    positive
3    positive
Name: overall, dtype: object

now i am gonna save these two columns into on dataset


In [25]:
df = df.rename(columns={
    'reviewText' : 'review',
    'overall':'sentiment'
})

df = df.drop(columns=['Unnamed: 0'])
df.head()


,review,sentiment
0,No issues.,positive
1,"Purchased this for my device, it worked as adv...",positive
2,it works as expected. I should have sprung for...,positive
3,This think has worked out great.Had a diff. br...,positive
4,"Bought it with Retail Packaging, arrived legit...",positive


In [26]:
df.to_csv('amazon_review_dataset.csv',index=False)

In [27]:
df = pd.read_csv('amazon_review_dataset.csv')

df.head()

,review,sentiment
0,No issues.,positive
1,"Purchased this for my device, it worked as adv...",positive
2,it works as expected. I should have sprung for...,positive
3,This think has worked out great.Had a diff. br...,positive
4,"Bought it with Retail Packaging, arrived legit...",positive


# EDA

In [47]:
# shape 
print(df.shape,"\n")

# info 
print(df.info(),'\n')

# duplicates
print(df.duplicated().sum(),'\n')

# drop duplicates
print(df.drop_duplicates(inplace=True),'\n')

# value counts of sentiment
print(df['sentiment'].value_counts(),'\n')

# null values
print(df.isna().sum(),'\n')

# drop null values
print(df.dropna(inplace=True),'\n')



(4912, 2) 

<class 'pandas.core.frame.DataFrame'>
Index: 4912 entries, 0 to 4914
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     4912 non-null   object
 1   sentiment  4912 non-null   object
dtypes: object(2)
memory usage: 115.1+ KB
None 

0 

None 

sentiment
positive    4446
negative     466
Name: count, dtype: int64 

review       0
sentiment    0
dtype: int64 

None 



In [49]:
!pip install nltk

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ------------- -------------------------- 0.5/1.6 MB 4.2 MB/s eta 0:00:01
   --------------------------------- ------ 1.3/1.6 MB 3.5 MB/s eta 0:00:01
   --------------------------------- ------ 1.3/1.6 MB 3.5 MB/s eta 0:00:01
   ---------------------------------------- 1.6/1.6 MB 2.4 MB/s eta 0:00:00



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: C:\Users\lokes\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [50]:

import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\lokes\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\lokes\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [51]:
stop_words = set(stopwords.words('english'))
lemmatizer  = WordNetLemmatizer()

In [ ]:

def normalize_text(text):

    # 1. changing text to lower case
    text = str(text).lower()

    # 2. Remove urls
    text = re.sub(r'https?://\s+|www\.\S+','',text)

    # remove html tags
    text = re.sub(r'<.*>','',text)

    # 4. reomve emojis and special characters (keep only a-z spaces)
    text = re.sub(r'[^a-z\s]','',text)

    # 5. Remove extra whitespaces

    text = " ".join(text.split())
    
    return text

def remove_noise(word_list):
    clean_tokens = [re.sub(r'[^A-Za-z0-9]+','',word) for word in word_list]

    return [word for word in clean_tokens if word]


def remove_stopwords(word_list):
    filtered_tokens = [word for word in word_list if word not in stop_words]
    return filtered_tokens

def lemmatize_text(tokens):
    return [lemmatizer.lemmatize(word) for word in tokens]


df['review'] = df['review'].apply(normalize_text)
df['review'] = df['review'].apply(word_tokenize)
df['review'] = df['review'].apply(remove_noise)
df['review'] = df['review'].apply(remove_stopwords)
df['review'] = df['review'].apply(lemmatize_text)

print(df['review'].iloc[0:4])

df["review"] = df['review'].apply(lambda x : ' '.join(x))



0                                              [issue]
1    [purchased, device, worked, advertised, never,...
2    [work, expected, sprung, higher, capacity, thi...
3    [think, worked, greathad, diff, bran, gb, card...
Name: review, dtype: object


In [55]:
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import(
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    classification_report
)

In [56]:
x_text = df['review']
y = df['sentiment']

In [57]:
print(len(x_text))
print(len(y))

4912
4912


In [62]:
vectorizer = TfidfVectorizer(
    min_df= 5,
    max_df=0.9,
    ngram_range=(1,2)
)

In [63]:
x = vectorizer.fit_transform(x_text)

In [64]:
print('Shape of the vectorizer matrix :\n',x.shape)

Shape of the vectorizer matrix :
 (4912, 4725)


In [66]:
# total no of features
print(len(vectorizer.get_feature_names_out()))

4725


In [68]:
x_train,x_test,y_train,y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [76]:
print("shape of the x_test : \n",x_test.shape)
print("shape of the x_train : \n",x_train.shape)
print("shape of the y_train : \n",y_train.shape)
print("shape of the y_test : \n",y_test.shape)

shape of the x_test : 
 (983, 4725)
shape of the x_train : 
 (3929, 4725)
shape of the y_train : 
 (3929,)
shape of the y_test : 
 (983,)


In [77]:
print(y_train.value_counts()['positive']/len(y_train))
print(y_train.value_counts()['negative']/len(y_train))
print(y_test.value_counts()['negative']/len(y_test))
print(y_test.value_counts()['positive']/len(y_test))

0.9050649020106898
0.09493509798931025
0.09460834181078331
0.9053916581892166


In [78]:
import joblib

joblib.dump(vectorizer,'tfidf_vectorizer.pkl')

['tfidf_vectorizer.pkl']

In [79]:
lr_model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000
)

In [80]:
lr_model.fit(x_train,y_train)

LogisticRegression(class_weight='balanced', max_iter=1000)

In [82]:
cv_scores = cross_val_score(
    lr_model,
    x_train,
    y_train,
    cv=5,
    scoring='f1_weighted'
)

In [83]:
print("Fold Scores:", cv_scores)
print("Average F1:", cv_scores.mean())
print("Std Dev:", cv_scores.std())

Fold Scores: [0.92800458 0.92993516 0.92519699 0.93388857 0.94609234]
Average F1: 0.932623524583289
Std Dev: 0.00730376174115932


In [85]:
y_pred = lr_model.predict(x_test)

In [86]:
accuracy = accuracy_score(
    y_test,y_pred
)

precision = precision_score(
    y_test,y_pred,
    average='weighted'
)

recall = recall_score(
    y_test,y_pred,
    average='weighted'
)

f1_score = f1_score(
    y_test,y_pred,
    average='weighted'
)

cl_report = classification_report(
    y_test,y_pred
)

In [87]:
print("accuracy : ",accuracy,'\n')
print("precision : ",precision,'\n')
print("recall : ",recall,'\n')
print("f1_score : ",f1_score,'\n')
print("classification report : ",cl_report,'\n')

accuracy :  0.9114954221770092 

precision :  0.9264238173300161 

recall :  0.9114954221770092 

f1_score :  0.9173276705873911 

classification report :                precision    recall  f1-score   support

    negative       0.52      0.71      0.60        93
    positive       0.97      0.93      0.95       890

    accuracy                           0.91       983
   macro avg       0.75      0.82      0.78       983
weighted avg       0.93      0.91      0.92       983
 



In [88]:
joblib.dump(
    lr_model,
    "sentiment_model.pkl"
)

['sentiment_model.pkl']

In [89]:
review = [
    "This product is amazing and battery life is excellent"
]

review_vector = vectorizer.transform(review)

prediction = lr_model.predict(
    review_vector
)

print(prediction)

['positive']
